In [0]:
CREATE SCHEMA IF NOT EXISTS hvacapp_dev3.master;

-- Create site table
CREATE OR REPLACE TABLE hvacapp_dev3.master.site (
  site_id STRING NOT NULL,
  site_name STRING NOT NULL,
  country STRING,
  state_region STRING,
  city STRING,
  timezone STRING,
  site_status STRING,
  created_ts TIMESTAMP,
  updated_ts TIMESTAMP
);

INSERT INTO hvacapp_dev3.master.site
VALUES
('SITE_001', 'Cyberjaya Demo Site', 'Malaysia', 'Selangor', 'Cyberjaya', 'Asia/Kuala_Lumpur', 'ACTIVE', current_timestamp(), current_timestamp());

SELECT * FROM hvacapp_dev3.master.site;

-- Create building table
CREATE OR REPLACE TABLE hvacapp_dev3.master.building (
  building_id STRING NOT NULL,
  site_id STRING NOT NULL,
  building_name STRING NOT NULL,
  building_type STRING,
  floor_count INT,
  gross_area_sqm DOUBLE,
  building_status STRING,
  created_ts TIMESTAMP,
  updated_ts TIMESTAMP
);

INSERT INTO hvacapp_dev3.master.building
VALUES
('BLDG_001', 'SITE_001', 'Main Building', 'Office', 10, 12000.0, 'ACTIVE', current_timestamp(), current_timestamp());

SELECT * FROM hvacapp_dev3.master.building;

-- Create equipment table
CREATE OR REPLACE TABLE hvacapp_dev3.master.equipment (
  equipment_id STRING NOT NULL,
  building_id STRING NOT NULL,
  site_id STRING NOT NULL,
  equipment_name STRING NOT NULL,
  equipment_type STRING NOT NULL,
  manufacturer STRING,
  model_number STRING,
  serial_number STRING,
  install_date DATE,
  equipment_status STRING,
  parent_equipment_id STRING,
  created_ts TIMESTAMP,
  updated_ts TIMESTAMP
);

INSERT INTO hvacapp_dev3.master.equipment
VALUES
('EQ_001', 'BLDG_001', 'SITE_001', 'Chiller-01', 'CHILLER', 'Carrier', '30XW', 'SN-CH-001', DATE '2024-01-15', 'ACTIVE', NULL, current_timestamp(), current_timestamp()),
('EQ_002', 'BLDG_001', 'SITE_001', 'CHW Pump-01', 'PUMP', 'Grundfos', 'TP-100', 'SN-PMP-001', DATE '2024-01-15', 'ACTIVE', 'EQ_001', current_timestamp(), current_timestamp());

SELECT * FROM hvacapp_dev3.master.equipment;

-- Create sensor table
CREATE OR REPLACE TABLE hvacapp_dev3.master.sensor (
  sensor_id STRING NOT NULL,
  equipment_id STRING NOT NULL,
  site_id STRING NOT NULL,
  building_id STRING NOT NULL,
  sensor_name STRING NOT NULL,
  metric_name STRING NOT NULL,
  unit STRING NOT NULL,
  data_type STRING,
  min_valid_value DOUBLE,
  max_valid_value DOUBLE,
  sensor_status STRING,
  created_ts TIMESTAMP,
  updated_ts TIMESTAMP
);

INSERT INTO hvacapp_dev3.master.sensor
VALUES
('SNS_001', 'EQ_001', 'SITE_001', 'BLDG_001', 'Chiller-01 Supply Temp Sensor', 'chw_supply_temp_c', 'C', 'DOUBLE', 3.0, 15.0, 'ACTIVE', current_timestamp(), current_timestamp()),
('SNS_002', 'EQ_001', 'SITE_001', 'BLDG_001', 'Chiller-01 Return Temp Sensor', 'chw_return_temp_c', 'C', 'DOUBLE', 5.0, 25.0, 'ACTIVE', current_timestamp(), current_timestamp()),
('SNS_003', 'EQ_001', 'SITE_001', 'BLDG_001', 'Chiller-01 Power Meter', 'hvac_power_kw', 'kW', 'DOUBLE', 0.0, 2000.0, 'ACTIVE', current_timestamp(), current_timestamp()),
('SNS_004', 'EQ_002', 'SITE_001', 'BLDG_001', 'Pump-01 Flow Sensor', 'chw_flow_lpm', 'LPM', 'DOUBLE', 0.0, 5000.0, 'ACTIVE', current_timestamp(), current_timestamp()),
('SNS_005', 'EQ_002', 'SITE_001', 'BLDG_001', 'Pump-01 Speed Sensor', 'pump_speed_pct', '%', 'DOUBLE', 0.0, 100.0, 'ACTIVE', current_timestamp(), current_timestamp());

SELECT * FROM hvacapp_dev3.master.sensor;

-- Create control_point table
CREATE OR REPLACE TABLE hvacapp_dev3.master.control_point (
  control_point_id STRING NOT NULL,
  equipment_id STRING NOT NULL,
  site_id STRING NOT NULL,
  building_id STRING NOT NULL,
  control_point_name STRING NOT NULL,
  control_metric_name STRING NOT NULL,
  unit STRING NOT NULL,
  min_allowed_value DOUBLE,
  max_allowed_value DOUBLE,
  default_value DOUBLE,
  control_mode STRING,
  control_status STRING,
  created_ts TIMESTAMP,
  updated_ts TIMESTAMP
);

INSERT INTO hvacapp_dev3.master.control_point
VALUES
('CP_001', 'EQ_001', 'SITE_001', 'BLDG_001', 'Chiller Supply Temp Setpoint', 'chw_supply_temp_setpoint_c', 'C', 6.0, 10.0, 9.0, 'STATIC', 'ACTIVE', current_timestamp(), current_timestamp()),
('CP_002', 'EQ_002', 'SITE_001', 'BLDG_001', 'Pump Speed Setpoint', 'pump_speed_setpoint_pct', '%', 30.0, 100.0, 75.0, 'STATIC', 'ACTIVE', current_timestamp(), current_timestamp());

SELECT * FROM hvacapp_dev3.master.control_point;

-- site
--  └── building
--       └── equipment
--            ├── sensor
--            └── control_point

-- Example:
-- SITE_001 = Cyberjaya Demo Site
-- BLDG_001 = Main Building
-- EQ_001 = Chiller-01
-- SNS_001 = Supply Temp Sensor
-- CP_001 = Supply Temp Setpoint

-- Test Joined View
SELECT
  s.site_name,
  b.building_name,
  e.equipment_name,
  e.equipment_type,
  sn.sensor_name,
  sn.metric_name,
  cp.control_point_name,
  cp.control_metric_name
FROM hvacapp_dev3.master.equipment e
LEFT JOIN hvacapp_dev3.master.site s
  ON e.site_id = s.site_id
LEFT JOIN hvacapp_dev3.master.building b
  ON e.building_id = b.building_id
LEFT JOIN hvacapp_dev3.master.sensor sn
  ON e.equipment_id = sn.equipment_id
LEFT JOIN hvacapp_dev3.master.control_point cp
  ON e.equipment_id = cp.equipment_id;

-- At this point, I have created the foundation for the whole HVAC simulation:

-- where the site is
-- what building is inside it
-- what equipment exists
-- what sensors measure it
-- what control points can be adjusted

-- Without this, later telemetry data will have no proper structure.